# 🔪 RingCutter — Data Exploration
## Notebook 1: Understanding Our Synthetic Banking Data

### Step 1: Import Libraries

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import plotly.express as px
import plotly.graph_objects as go
from collections import Counter
import warnings
warnings.filterwarnings('ignore')

# Set display options
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 50)
print('Libraries loaded ✅')

### Step 2: Load Data

In [ ]:
# Load accounts data
accounts_df = pd.read_csv('../data/synthetic_accounts.csv')
print(f'Accounts: {len(accounts_df)} rows')
print(f'Columns: {list(accounts_df.columns)}')
accounts_df.head()

In [ ]:
# Load transactions data
txns_df = pd.read_csv('../data/synthetic_transactions.csv')
txns_df['timestamp'] = pd.to_datetime(txns_df['timestamp'])
print(f'Transactions: {len(txns_df)} rows')
print(f'Columns: {list(txns_df.columns)}')
txns_df.head()

### Step 3: Account Analysis

In [ ]:
# Normal vs Mule accounts
print('=== ACCOUNT BREAKDOWN ===')
print(f'Total accounts: {len(accounts_df)}')
print(f'Normal accounts: {len(accounts_df[accounts_df["is_mule"] == 0])}')
print(f'Mule accounts: {len(accounts_df[accounts_df["is_mule"] == 1])}')
print(f'Mule percentage: {len(accounts_df[accounts_df["is_mule"] == 1]) / len(accounts_df) * 100:.1f}%')
print(f'Number of rings: {accounts_df[accounts_df["mule_ring_id"] >= 0]["mule_ring_id"].nunique()}')

In [ ]:
# Compare normal vs mule account characteristics
normal = accounts_df[accounts_df['is_mule'] == 0]
mule = accounts_df[accounts_df['is_mule'] == 1]

print('\n=== COMPARISON: NORMAL vs MULE ===')
print(f'{"Feature":<25} {"Normal":>15} {"Mule":>15}')
print('-' * 55)
print(f'{"Avg Account Age (days)":<25} {normal["account_age_days"].mean():>15.0f} {mule["account_age_days"].mean():>15.0f}')
print(f'{"Avg Balance (₹)":<25} {normal["avg_monthly_balance"].mean():>15,.0f} {mule["avg_monthly_balance"].mean():>15,.0f}')
print(f'{"Avg Age (years)":<25} {normal["age"].mean():>15.1f} {mule["age"].mean():>15.1f}')
print(f'{"MIN_KYC %":<25} {(normal["kyc_status"] == "MIN_KYC").mean() * 100:>14.1f}% {(mule["kyc_status"] == "MIN_KYC").mean() * 100:>14.1f}%')

In [ ]:
# Account Age Distribution: Normal vs Mule
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].hist(normal['account_age_days'], bins=50, color='#00CC66', alpha=0.7, label='Normal')
axes[0].hist(mule['account_age_days'], bins=50, color='#FF4B4B', alpha=0.7, label='Mule')
axes[0].set_title('Account Age Distribution')
axes[0].set_xlabel('Account Age (days)')
axes[0].set_ylabel('Count')
axes[0].legend()

axes[1].hist(normal['avg_monthly_balance'], bins=50, color='#00CC66', alpha=0.7, label='Normal')
axes[1].hist(mule['avg_monthly_balance'], bins=50, color='#FF4B4B', alpha=0.7, label='Mule')
axes[1].set_title('Balance Distribution')
axes[1].set_xlabel('Average Monthly Balance (₹)')
axes[1].set_ylabel('Count')
axes[1].legend()

plt.tight_layout()
plt.savefig('../data/account_distribution.png', dpi=150, bbox_inches='tight')
plt.show()
print('Chart saved ✅')

### Step 4: Transaction Analysis

In [ ]:
# Transaction breakdown
print('=== TRANSACTION BREAKDOWN ===')
print(f'Total transactions: {len(txns_df)}')
print(f'Normal transactions: {len(txns_df[txns_df["is_fraud"] == 0])}')
print(f'Fraud transactions: {len(txns_df[txns_df["is_fraud"] == 1])}')
print(f'Fraud percentage: {len(txns_df[txns_df["is_fraud"] == 1]) / len(txns_df) * 100:.1f}%')

print('\n=== CHANNEL DISTRIBUTION ===')
print(txns_df['channel'].value_counts())

print('\n=== AMOUNT STATISTICS ===')
print(txns_df['amount'].describe())

In [ ]:
# Channel distribution for fraud vs normal
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

normal_txns = txns_df[txns_df['is_fraud'] == 0]
fraud_txns = txns_df[txns_df['is_fraud'] == 1]

normal_txns['channel'].value_counts().plot(kind='bar', ax=axes[0], color='#00CC66')
axes[0].set_title('Normal Transactions by Channel')
axes[0].set_ylabel('Count')

fraud_txns['channel'].value_counts().plot(kind='bar', ax=axes[1], color='#FF4B4B')
axes[1].set_title('Fraud Transactions by Channel')
axes[1].set_ylabel('Count')

plt.tight_layout()
plt.savefig('../data/channel_distribution.png', dpi=150, bbox_inches='tight')
plt.show()
print('Chart saved ✅')

In [ ]:
# Amount distribution: Normal vs Fraud
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].hist(normal_txns['amount'], bins=50, color='#00CC66', alpha=0.7)
axes[0].set_title('Normal Transaction Amounts')
axes[0].set_xlabel('Amount (₹)')

axes[1].hist(fraud_txns['amount'], bins=50, color='#FF4B4B', alpha=0.7)
axes[1].set_title('Fraud Transaction Amounts')
axes[1].set_xlabel('Amount (₹)')

plt.tight_layout()
plt.savefig('../data/amount_distribution.png', dpi=150, bbox_inches='tight')
plt.show()
print('Chart saved ✅')

### Step 5: Mule Ring Analysis

In [ ]:
# Analyze each mule ring
print('=== MULE RING DETAILS ===')
print(f'{"Ring ID":<10} {"Members":<10} {"Cities":<30} {"Shared Devices":<15}')
print('-' * 65)

for ring_id in sorted(accounts_df[accounts_df['mule_ring_id'] >= 0]['mule_ring_id'].unique()):
    ring = accounts_df[accounts_df['mule_ring_id'] == ring_id]
    cities = ', '.join(ring['city'].unique())
    # Count shared devices
    device_counts = ring['primary_device_id'].value_counts()
    shared = (device_counts > 1).sum()
    print(f'{ring_id:<10} {len(ring):<10} {cities:<30} {shared:<15}')

In [ ]:
# Hourly transaction distribution
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

normal_txns['hour'].hist(bins=24, ax=axes[0], color='#00CC66', alpha=0.7)
axes[0].set_title('Normal Transactions by Hour')
axes[0].set_xlabel('Hour of Day')

fraud_txns['hour'].hist(bins=24, ax=axes[1], color='#FF4B4B', alpha=0.7)
axes[1].set_title('Fraud Transactions by Hour')
axes[1].set_xlabel('Hour of Day')

plt.tight_layout()
plt.savefig('../data/hourly_distribution.png', dpi=150, bbox_inches='tight')
plt.show()
print('Chart saved ✅')

### Step 6: Key Findings Summary

In [ ]:
print('=' * 60)
print('  KEY FINDINGS FROM DATA EXPLORATION')
print('=' * 60)
print()
print('1. MULE ACCOUNTS are significantly NEWER')
print(f'   Normal avg age: {normal["account_age_days"].mean():.0f} days')
print(f'   Mule avg age:   {mule["account_age_days"].mean():.0f} days')
print()
print('2. MULE ACCOUNTS have LOWER balances')
print(f'   Normal avg balance: ₹{normal["avg_monthly_balance"].mean():,.0f}')
print(f'   Mule avg balance:   ₹{mule["avg_monthly_balance"].mean():,.0f}')
print()
print('3. MULE ACCOUNTS more likely to have MIN_KYC')
print(f'   Normal MIN_KYC: {(normal["kyc_status"] == "MIN_KYC").mean() * 100:.1f}%')
print(f'   Mule MIN_KYC:   {(mule["kyc_status"] == "MIN_KYC").mean() * 100:.1f}%')
print()
print('4. FRAUD transactions cluster in ₹15,000-₹49,900 range')
print(f'   Fraud avg amount: ₹{fraud_txns["amount"].mean():,.0f}')
print(f'   Normal avg amount: ₹{normal_txns["amount"].mean():,.0f}')
print()
print('5. DEVICE SHARING exists within mule rings')
print('   This is a key detection signal for the GNN')
print()
print('✅ Data exploration complete!')